# SAFEWORK AI — latih model APD di GPU Colab (dari VS Code atau browser)

Notebook ini melatih YOLO11n dari dataset **Construction-PPE**, lalu menghasilkan `best.pt` untuk dipakai di aplikasi:

`python -m safework --weights runs\ppe-v1\weights\best.pt`

**Dari VS Code (disarankan):** pasang ekstensi **Colab** (publisher: Google) dan **Jupyter** → buka file ini →
*Select Kernel → Colab → New Colab Server* → pilih GPU (mis. T4) → masuk dengan akun Google → *Run All*.

**Dari browser:** unggah ke colab.research.google.com → *Runtime → Change runtime type → T4 GPU* → *Run all*.

Atur sel konfigurasi di bawah sesuai lingkungan yang kamu pakai.

> Catatan jujur: notebook ini belum pernah dijalankan oleh pembuatnya (tanpa akses internet dan GPU).
> Bila ada galat atau perilaku ekstensi Colab berubah, salin pesannya ke chat.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Pilih salah satu: "vscode" (ekstensi Colab di VS Code) atau "colab-web" (colab.research.google.com)
LINGKUNGAN = "vscode"
assert LINGKUNGAN in ("vscode", "colab-web"), 'LINGKUNGAN harus "vscode" atau "colab-web"'

In [ ]:
!nvidia-smi
import torch
print("CUDA tersedia:", torch.cuda.is_available())

In [ ]:
!pip -q install -U ultralytics

## 1. Latih dari Construction-PPE

Dataset diunduh otomatis ke server Colab (jaringan Google, jadi tidak memakai kuota internetmu).
Kelasnya: helmet, gloves, vest, boots, goggles, Person, none, no_helmet, no_goggle, no_gloves, no_boots.
Tidak ada `no_vest` maupun `mask` (itu nanti dari foto lokal). Lisensi dataset dan pustaka Ultralytics: AGPL-3.0.

In [ ]:
from pathlib import Path
from ultralytics import YOLO

model = YOLO("yolo11n.pt")
model.train(
    data="construction-ppe.yaml",   # diunduh otomatis saat pertama kali
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,                    # berhenti lebih awal bila tidak membaik selama 20 epoch
    project="runs",
    name="ppe-v1",
    exist_ok=True,
)
best = Path(model.trainer.best)
print("Bobot terbaik:", best)

In [ ]:
# Evaluasi pada data uji yang tidak dipakai saat melatih
best_model = YOLO(best)
try:
    metrics = best_model.val(data="construction-ppe.yaml", split="test")
except Exception as exc:
    print("Split test tidak tersedia, memakai validation:", exc)
    metrics = best_model.val(data="construction-ppe.yaml")

print(f"mAP50 = {metrics.box.map50:.3f} | mAP50-95 = {metrics.box.map:.3f}")
for idx, ap50 in zip(metrics.box.ap_class_index, metrics.box.ap50):
    print(f"{metrics.names[int(idx)]:12s} AP50 = {ap50:.3f}")

In [ ]:
# Confusion matrix dan kurva latihan. Perhatikan: seberapa sering no_* muncul padahal APD dipakai (alarm palsu)?
from IPython.display import Image, display

run_dir = best.parent.parent
for name in ["confusion_matrix_normalized.png", "results.png"]:
    path = run_dir / name
    if path.exists():
        display(Image(filename=str(path)))

## Mengambil `best.pt` (ukurannya beberapa MB)

* **Browser (`colab-web`)**: sel di bawah langsung mengunduhnya lewat `files.download`.
* **VS Code (`vscode`)**: ekstensi Colab belum mendukung `files.download()`. Sel di bawah menyalinnya ke Google Drive:
  buka Command Palette (`Ctrl+Shift+P`) → **Colab: Mount Google Drive to Server...**, jalankan sel yang ditambahkan
  (izinkan akses Drive), lalu jalankan sel di bawah. Unduh dari drive.google.com, folder `safework`.
  Alternatif: lihat folder `/content/runs/` lewat panel Colab di Activity Bar (tampilan *Contents*).

In [ ]:
import shutil
from pathlib import Path


def ambil_bobot(path, nama_tujuan, drive_root=Path("/content/drive/MyDrive")):
    """Siapkan berkas bobot supaya bisa kamu ambil, sesuai lingkungan."""
    path = Path(path)
    print("Lokasi di server Colab:", path.resolve())
    if LINGKUNGAN == "colab-web":
        from google.colab import files
        files.download(str(path))
        return
    # Ekstensi Colab di VS Code: files.download() belum didukung, jadi salin ke Google Drive.
    if drive_root.exists():
        tujuan = drive_root / "safework"
        tujuan.mkdir(parents=True, exist_ok=True)
        shutil.copy(path, tujuan / nama_tujuan)
        print("Tersalin ke Google Drive:", tujuan / nama_tujuan, "(unduh dari drive.google.com)")
    else:
        print("Drive belum di-mount. Buka Command Palette (Ctrl+Shift+P) -> 'Colab: Mount Google Drive to Server...',\n"
              "jalankan sel yang ditambahkan, lalu jalankan sel ini lagi.\n"
              "Alternatif: ambil berkas lewat panel Colab di Activity Bar (tampilan Contents), folder /content/runs/.")


ambil_bobot(best, "best_ppe_v1.pt")

## 2. Nanti: latih ulang dengan foto lokal

Setelah foto lokalmu dianotasi (CVAT atau Label Studio, ekspor format YOLO), kemas folder berisi
`images/train`, `images/val`, `labels/train`, `labels/val` menjadi `dataset_lokal.zip`, lalu unggah ke `/content`:

* **VS Code:** klik kanan `dataset_lokal.zip` di Explorer → **Upload to Colab**.
* **Browser:** panel *Files* di kiri → unggah.

**Urutan kelas di YAML di bawah harus sama dengan id kelas pada label hasil anotasimu.**
Sel ini otomatis dilewati bila zip belum ada.

In [ ]:
import zipfile

zip_path = Path("/content/dataset_lokal.zip")
if not zip_path.exists():
    print("Dilewati: unggah dataset_lokal.zip ke /content setelah foto lokalmu dianotasi.")
else:
    zipfile.ZipFile(zip_path).extractall("/content/dataset_lokal")
    Path("/content/local.yaml").write_text("""path: /content/dataset_lokal
train: images/train
val: images/val
names:
  0: person
  1: helmet
  2: vest
  3: gloves
  4: boots
  5: goggles
  6: mask
  7: no_helmet
  8: no_vest
  9: no_gloves
  10: no_boots
  11: no_goggle
  12: no_mask
""")
    model2 = YOLO(best)             # lanjut dari bobot v1 (fine-tuning)
    model2.train(data="/content/local.yaml", epochs=60, imgsz=640, batch=16, patience=15,
                 project="runs", name="ppe-v2", exist_ok=True)
    best2 = Path(model2.trainer.best)
    print("Bobot terbaik v2:", best2)
    ambil_bobot(best2, "best_ppe_v2.pt")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Pakai di laptop

1. Simpan `best.pt` hasil unduhan ke `runs\ppe-v1\weights\best.pt` di folder `safework-ai`.
2. Cetak nama kelas model: `python scripts/benchmark_fps.py --weights runs\ppe-v1\weights\best.pt --frames 20`.
   Bila ada nama yang tidak dikenal, tambahkan alias di `safework/labels.py` (ALIASES).
3. Jalankan `python -m safework --weights runs\ppe-v1\weights\best.pt` (tambah `--serial COM5` bila ESP32 sudah ada).